# 03 · From PyTorch to SpiNNaker2: the direction task, trained and transferred

In notebook 02 you built a direction detector for four pixels by hand. Here a small network learns the same task in snnTorch (a spiking-network library for PyTorch), is exported to NIR and converted into a py-spinnaker2 network, the same kind of populations and projections you built yourself. Then you check, spike for spike, that the converted network computes exactly what the trained one does, and break the conversion on purpose to see what goes wrong without any error message. Runs on Brian2; cluster participants run it on the chip in section 8. Notebook 04 then scales the task up to an 8 × 8 sensor with noise.

**Try** marks exercises, **Discuss** marks questions we take up together. Write your answers in the cells below the questions.

Adapted in parts from the vendor's [tutorial 08, "The NIR workflow"](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/08_nir.ipynb) (SpiNNcloud, Apache-2.0). Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch
import torch.nn as nn
import torch.nn.functional as F
from snntorch import surrogate, utils
from snntorch.export_nir import export_to_nir
from spinnaker2 import brian2_sim, s2_nir

plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

brian2 = brian2_sim.Brian2Backend()  # sections 1 to 7 run on Brian2, section 8 switches to the chip
torch.manual_seed(0)  # the same training result on every run

## 1. The task from notebook 02, as a dataset

The same input: a dot moves across four pixels, to the right or to the left, fast (`step=1`) or slowly (`step=2`). To learn, the network needs examples with the right answer (the label: 0 = right, 1 = left). The dot starts at different time steps (2 to 8), so the network cannot simply learn one fixed moment. Two test sets it never sees during training: later start times, and inputs where one pixel misses the dot.

In [ ]:
N_PIXELS, STEPS = 4, 20


def moving_dot(direction, step=1, start=5):
    """Spike times of the pixels, {pixel: [time step]}, for a dot moving "right" or "left" (as in notebook 02)."""
    order = range(N_PIXELS) if direction == "right" else reversed(range(N_PIXELS))
    return {pixel: [start + k * step] for k, pixel in enumerate(order)}


def without(spikes, pixel):
    """The same input with one pixel missing the dot."""
    return {p: ([] if p == pixel else times) for p, times in spikes.items()}


def to_tensor(spikes):
    """Spike times -> tensor of shape (time steps, pixels) with 1 where a pixel fires."""
    x = torch.zeros(STEPS, N_PIXELS)
    for pixel, times in spikes.items():
        for t in times:
            x[t, pixel] = 1
    return x


def make_set(examples):
    """List of (direction, spikes) -> inputs (time steps, examples, pixels) and labels (0 = right, 1 = left)."""
    x = torch.stack([to_tensor(spikes) for _, spikes in examples], dim=1)
    y = torch.tensor([0 if direction == "right" else 1 for direction, _ in examples])
    return x, y


train_examples = [(d, moving_dot(d, s, t0)) for d in ("right", "left") for s in (1, 2) for t0 in range(2, 9)]
later_examples = [(d, moving_dot(d, s, t0)) for d in ("right", "left") for s in (1, 2) for t0 in range(9, 13)]
missing_examples = [(d, without(moving_dot(d, s), p)) for d in ("right", "left") for s in (1, 2) for p in range(N_PIXELS)]
x_train, y_train = make_set(train_examples)
print(f"{len(train_examples)} training examples, {len(later_examples)} with later start times, {len(missing_examples)} with one pixel missing")

## 2. The network in snnTorch

Four input pixels, a hidden layer of 8 LIF neurons and two output neurons, one per direction; the answer is the output neuron that fires more often. In snnTorch, `nn.Linear` holds the weights (like `W` in notebook 02) and `snn.Leaky` is a layer of LIF neurons: `beta` is the share of the potential kept per step, the same as `alpha_decay`, and `reset_mechanism="zero"` resets to 0 after a spike. There are no delays: the network has to learn the order of the spikes with its weights alone.

<details><summary>The update rule of <code>snn.Leaky</code></summary>

Per time step, `snn.Leaky` computes $U[t+1] = \beta\, U[t] + I_\text{in}[t+1]$, where $I_\text{in}$ is the output of the preceding `nn.Linear`. The neuron spikes when $U$ exceeds the threshold (1 by default); with `reset_mechanism="zero"` it is then set to 0. This is the update of the `lif` neuron of notebook 01, with `beta` in the role of `alpha_decay`. `init_hidden=True` lets each layer keep its potential internally, so `nn.Sequential` works. Source: the `snn.Leaky` docstring and [snnTorch tutorial 2](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_2.html).

</details>

In [ ]:
beta = 0.5  # share of the potential kept per step
spike_grad = surrogate.fast_sigmoid()  # smooth stand-in for the step function, used only for training
net = nn.Sequential(
    nn.Linear(N_PIXELS, 8, bias=False),  # weights pixels -> hidden
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad),
    nn.Linear(8, 2, bias=False),  # weights hidden -> outputs
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad, output=True),
)


def run_snntorch(x):
    """Run the network step by step over inputs (time steps, examples, pixels); return hidden and output spikes over time."""
    utils.reset(net)  # start from potential 0
    hidden, output = [], []
    for x_t in x:  # one step after the other, like the chip
        h = net[1](net[0](x_t))
        o, _ = net[3](net[2](h))
        hidden.append(h)
        output.append(o)
    return torch.stack(hidden), torch.stack(output)  # (time steps, examples, neurons)


print(net)

## 3. Train it

Training adjusts the weights so that, for every training example, the right output neuron fires more often than the wrong one: the loss is the cross-entropy of the two spike counts, as in a classifier. Because a spike is a step function without a useful gradient, snnTorch uses a smooth stand-in in the backward pass, the surrogate gradient (slides before this block). With 28 examples, training takes a few seconds.

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
for epoch in range(1, 301):
    counts = run_snntorch(x_train)[1].sum(dim=0)  # spike count of each output neuron
    loss = F.cross_entropy(counts, y_train)
    optimizer.zero_grad()
    loss.backward()  # backpropagation through time, with the surrogate gradient
    optimizer.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch}: loss {loss.item():.3f}")


def accuracy(examples):
    """Share of examples for which the right output neuron fires more often (a tie counts as wrong)."""
    x, y = make_set(examples)
    with torch.no_grad():
        counts = run_snntorch(x)[1].sum(dim=0)
    return ((counts.argmax(dim=1) == y) & (counts[:, 0] != counts[:, 1])).float().mean().item()


for name, examples in [("training", train_examples), ("later start times", later_examples), ("one pixel missing", missing_examples)]:
    print(f"{name:18s}: {accuracy(examples):.0%} correct")

## 4. Export to NIR

NIR (Neuromorphic Intermediate Representation) describes the network as a graph of building blocks: weight layers (`Linear`) and neuron groups (`LIF`), connected by edges. snnTorch runs the model once on an example input to record this graph; only the size of the input matters, not its values.

<details><summary>What NIR is, and why it exists</summary>

Every spiking framework and chip describes neurons in its own way, so a trained network used to be tied to one of them. NIR is a shared file format, comparable to ONNX for deep learning: each building block (a "primitive") is defined by a continuous-time equation, independent of any simulator's time step. The LIF primitive, for example, is

$$\tau\, \dot v = (v_\text{leak} - v) + R\, I, \qquad \text{spike if } v > v_\text{thr}.$$

Each framework exports to it, each chip imports from it and discretises the equations with its own step. [Pedersen et al. 2024 (Nature Communications)](https://doi.org/10.1038/s41467-024-52259-9) introduce NIR and show it with 7 simulators and 4 hardware platforms, SpiNNaker2 among them. The list of primitives: [NIR documentation](https://neuroir.org/docs/primitives/).

</details>

In [ ]:
nir_graph = export_to_nir(net, torch.zeros(N_PIXELS))
nir_graph.infer_types()  # fill in the input and output shapes of every node
s2_nir.model_summary(nir_graph)

## 5. Convert it into a py-spinnaker2 network

`s2_nir.from_nir` turns the NIR graph into `snn.Population` and `snn.Projection` objects, the same objects you built by hand in notebook 02. A `ConversionConfig` controls the translation. **Rule for models from snnTorch:** `dt = 1e-4`, integrator `FORWARD`, and the reset rule of the snnTorch model. Otherwise the chip runs a slightly different network, without any error message:

| Setting | Value for snnTorch | Why |
|---|---|---|
| `dt` | `1e-4` | snnTorch stores `beta` as a time constant $\tau = dt / (1 - \beta)$ with a fixed $dt = 10^{-4}$; only the same `dt` gives back `beta` |
| `integrator` | `FORWARD` | snnTorch computes $v[t+1] = \beta\, v[t] + \text{input}$, a forward Euler step. The py-spinnaker2 default `EXPONENTIAL` would give a decay of 0.607 instead of this network's 0.5 |
| `reset` | `ZERO` here | NIR does not store the reset rule, so it has to be set by hand to match `reset_mechanism` |

A fourth difference concerns the comparison: the chip delays every spike by one step per projection, snnTorch does not (section 6).

<details><summary>Forward and exponential, in numbers</summary>

snnTorch exports `beta` as a time constant $\tau = \Delta t / (1 - \beta)$ with $\Delta t = 10^{-4}$ s; for $\beta = 0.9$ that is $\tau = 10^{-3}$ s. py-spinnaker2 turns $\tau$ back into a decay factor per step:

- `FORWARD` ([forward Euler](https://en.wikipedia.org/wiki/Euler_method)): $\alpha = 1 - \Delta t / \tau = 0.9$, exactly snnTorch's `beta`
- `EXPONENTIAL` (exact solution of the decay): $\alpha = e^{-\Delta t / \tau} = e^{-0.1} \approx 0.905$

Both are valid ways to discretise the NIR equation; only the forward one reproduces what snnTorch computed. A different `dt` changes $\alpha$ as well: with `dt = 1e-3`, forward Euler gives $\alpha = 0$, a neuron without memory. Source: `convert_LIF` in py-spinnaker2's `s2_nir.py`.

</details>

<details><summary>What <code>scale_weights</code> does</summary>

NIR layers become `lif_no_delay` populations on the chip, with 8-bit integer weights (−127…127) and no synaptic delays (hence `conn_delay = 0`). `scale_weights = True` multiplies the weights of each layer so that the largest one becomes 127, and multiplies the thresholds by the same factor, so the network behaves the same apart from rounding. With `weight_scale_percentile` below 100, a percentile of the weights is mapped to 127 instead, which helps when a few outliers would otherwise squeeze all other weights into a few integer values. Source: [py-spinnaker2 API docs, s2_nir](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/s2_nir-api.html).

</details>

In [ ]:
def make_config(integrator=s2_nir.IntegratorMethod.FORWARD, reset=s2_nir.ResetMethod.ZERO):
    cfg = s2_nir.ConversionConfig()
    cfg.dt = 1e-4  # the rule for snnTorch models
    cfg.integrator = integrator
    cfg.reset = reset
    cfg.output_record = ["spikes"]
    cfg.conn_delay = 0
    cfg.scale_weights = True  # scale the float weights into the chip's integer range
    return cfg


s2_net, inputs, outputs = s2_nir.from_nir(nir_graph, make_config())
for pop in s2_net.populations:
    print(f"population {pop.name}: {pop.size} neurons, model {pop.app_class.app_name}")
first = s2_net.projections[0]  # pixels -> hidden
print(f"\nprojection {first.pre.name} -> {first.post.name}, weights as a matrix (one row per hidden neuron, one column per pixel, like W in notebook 02):")
print(first.sparse_weights.toarray().T.astype(int))
print("the same weights in snnTorch:")
print(np.round(net[0].weight.detach().numpy(), 2))

These are the objects of notebook 02: a `spike_list` population for the pixels, two `lif_no_delay` populations and two projections with a weight per connection, printed above as a matrix like `W` in notebook 02. Two things are new. The weights are whole numbers: py-spinnaker2 scaled the trained float weights so that the largest becomes 127 and rounded them, because the synapse word of `lif_no_delay` stores 7 bits plus a sign (the hardware slides). And the threshold was scaled by the same factor, so that the network still computes the same.

<details><summary>Forward and exponential, in numbers</summary>

snnTorch exports `beta` as a time constant $\tau = \Delta t / (1 - \beta)$ with $\Delta t = 10^{-4}$ s; for $\beta = 0.5$ that is $\tau = 2 \cdot 10^{-4}$ s. py-spinnaker2 turns $\tau$ back into a decay factor per step: `FORWARD` gives $1 - \Delta t / \tau = 0.5$, exactly snnTorch's `beta`; the default `EXPONENTIAL` gives $e^{-\Delta t / \tau} = e^{-0.5} \approx 0.607$. The smaller `beta`, the larger the difference: for $\beta = 0.9$ it would be 0.905 against 0.9, often too small to change a spike. That is why this network uses $\beta = 0.5$.

</details>

## 6. Same network? Compare spike for spike

Run one input through the converted network on Brian2 and through the original in snnTorch, and compare every spike of the hidden and the output layer. On SpiNNaker2 each spike reaches the next layer one step later than in snnTorch, so the py-spinnaker2 spikes are shifted back by one step for the hidden layer and two for the output layer before comparing.

<details><summary>Why one step per projection</summary>

In snnTorch, a spike computed in step $t$ reaches the next layer in the same step $t$, because the whole forward pass runs within one step. On the chip, each layer runs on its own cores and a spike travels to the next core as a packet, which is processed in the following step (the `delay + 1` rule of hands-on 1, with `delay = 0`). A network with $L$ projections therefore produces its output $L$ steps later. For classification by spike count, this hardly matters; for spike-for-spike comparisons and for networks that react in real time, it does.

</details>

In [ ]:
def spikes_py_spinnaker2(spikes, cfg, backend=brian2):
    """Convert with cfg, run one input, return hidden and output spikes, shifted back by one step per projection."""
    s2_net, inputs, outputs = s2_nir.from_nir(nir_graph, cfg)
    inputs[0].params = spikes
    hidden = [p for p in s2_net.populations if p not in inputs and p not in outputs][0]
    hidden.record = ["spikes"]
    backend.run(s2_net, STEPS + 2)  # 2 extra steps for the delay of the two projections
    shift = lambda d, k: {i: [int(t) - k for t in ts] for i, ts in d.items() if len(ts)}
    return shift(hidden.get_spikes(), 1), shift(outputs[0].get_spikes(), 2)


def spikes_snntorch(spikes):
    """Hidden and output spike times of the original snnTorch model for one input."""
    with torch.no_grad():
        hidden, output = run_snntorch(to_tensor(spikes).unsqueeze(1))
    as_dict = lambda m: {i: torch.nonzero(m[:, 0, i]).flatten().tolist() for i in range(m.shape[2]) if m[:, 0, i].any()}
    return as_dict(hidden), as_dict(output)


def compare(cfg, label, examples):
    """Count the inputs for which hidden and output spikes are identical, and the right answers of the converted network."""
    same_hidden = same_output = right = 0
    for direction, spikes in examples:
        s2_h, s2_o = spikes_py_spinnaker2(spikes, cfg)
        t_h, t_o = spikes_snntorch(spikes)
        same_hidden += s2_h == t_h
        same_output += s2_o == t_o
        counts = [len(s2_o.get(i, [])) for i in range(2)]
        right += counts[0] != counts[1] and int(np.argmax(counts)) == (0 if direction == "right" else 1)
    n = len(examples)
    print(f"{label}: hidden layer identical in {same_hidden}/{n}, output layer in {same_output}/{n}; right answers {right}/{n}")


one_input = moving_dot("right")
s2_h, s2_o = spikes_py_spinnaker2(one_input, make_config())
t_h, t_o = spikes_snntorch(one_input)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (h, o, title) in zip(axes, [(t_h, t_o, "snnTorch"), (s2_h, s2_o, "py-spinnaker2 on Brian2 (shifted back)")]):
    for i, ts in h.items():
        ax.plot(ts, [i] * len(ts), "|", color="0.4", ms=12, mew=2)
    for i, ts in o.items():
        ax.plot(ts, [9 + i] * len(ts), "|", color="C1", ms=12, mew=2)
    ax.set(title=title, xlabel="time step", xlim=(0, STEPS), yticks=list(range(8)) + [9, 10],
           yticklabels=[f"hidden {i}" for i in range(8)] + ["out: right", "out: left"])
plt.tight_layout()
plt.show()

all_examples = train_examples + later_examples + missing_examples
compare(make_config(), "correct settings", all_examples)

## 7. Break it on purpose

None of the following mistakes raises an error: py-spinnaker2 converts and runs the network either way. Predict first, then run the line in the cell below and compare with your prediction.

**Try:**

1. Use the py-spinnaker2 default integrator, `EXPONENTIAL`. Which layer will differ first, the hidden or the output layer? Will the answers change?
2. Use the wrong reset rule, `SUBTRACT` instead of `ZERO`. Which mistake do you expect to be worse, this one or the integrator, and why?
3. Compare without shifting the spikes back: what would the comparison report, and why? (Reason about it, or change `k` in `spikes_py_spinnaker2` to 0 and run section 6 again.)

In [ ]:
compare(make_config(integrator=s2_nir.IntegratorMethod.EXPONENTIAL), "integrator EXPONENTIAL", all_examples)
compare(make_config(reset=s2_nir.ResetMethod.SUBTRACT), "reset SUBTRACT", all_examples)

*Your answers:*

1. …
2. …
3. …

**Discuss:**

- With the wrong settings, the spikes change but the answers hardly do. Which of these mistakes would you notice in a classifier that only counts output spikes, and which only in a spike-for-spike comparison? What does that mean for checking a converted network?
- Compare the trained network with your hand-built detector of notebook 02: what does each use to tell the direction?

*Your answers:*

- …
- …

## 8. Run it on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` and run the cell once per pair: the converted network runs for one input on one 48-chip board (about one minute) and is compared spike for spike with snnTorch.

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

if USE_HARDWARE:
    from spinnaker2 import hardware

    hw = hardware.SpiNNcloud48NodeBoard()
else:
    hw = brian2_sim.Brian2Backend()

print("Running on backend:", type(hw).__name__)

chip_h, chip_o = spikes_py_spinnaker2(moving_dot("left"), make_config(), backend=hw)
t_h, t_o = spikes_snntorch(moving_dot("left"))
print(f"{type(hw).__name__}: hidden layer {'identical' if chip_h == t_h else 'DIFFERENT'}, output layer {'identical' if chip_o == t_o else 'DIFFERENT'} to snnTorch")

## 9. What you built

The task of notebook 02, solved a second time: a network trained in snnTorch, with weights instead of hand-chosen delays, exported to NIR and converted into the same kind of populations and projections you built by hand. With the right settings, the converted network fires exactly the same spikes, shifted by one step per projection, with integer weights. With the wrong settings, it computes something slightly different without any warning, which the answers alone may not reveal.

Notebook 04 scales the task up: an 8 × 8 sensor, four directions, noise and lost events, and a network that has to generalise.

**Go deeper:** tutorial 08, sections 2 to 4 (the NIR primitives py-spinnaker2 supports, the same sanity check, settings for large trained models); [py-spinnaker2 user guide, NIR](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/qs-nir.html); workshop notes: [the conversion rule for snnTorch models](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).